In [108]:
import ee
import os
import json
import pandas as pd
import geopandas as gpd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np


In [3]:
PROJECT_ROOT = Path().resolve().parent

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

True

In [3]:
ee.Authenticate()



Successfully saved authorization token.


In [96]:
ee.Initialize(project=os.environ["EE_PROJECT"])

In [151]:
import importlib
import data_utils
import gee_utils

importlib.reload(data_utils)
importlib.reload(gee_utils)

<module 'gee_utils' from '/Users/bensutton/Projects/dissertation-code/src/gee_utils.py'>

In [152]:

sys.path.append(str(PROJECT_ROOT / "src"))

from gee_utils import create_images_for_all_locations, get_samples, export_patches, export_awei_p95_all_locations
from data_utils import make_padded_bbox_all_location, clean_labelled_data


In [91]:
# Want to upload the sites.json to ee and then create image collection for each, clip to the bbox, need to select the dates for each location
# Could save the location in the json or just use the date from the points? 

EE_PROJECT = os.environ["EE_PROJECT"]
site_fp = PROJECT_ROOT / "configs" / "sites.json"
label_fp = PROJECT_ROOT / "configs" / "labels.gpkg"
cleaned_label_fp = PROJECT_ROOT / "configs" / "cleaned_labels.gpkg"

In [36]:
clean_labelled_data(label_fp= label_fp, cleaned_label_fp = cleaned_label_fp)

Saved a cleaned version of /Users/bensutton/Projects/dissertation-code/configs/labels.gpkg as /Users/bensutton/Projects/dissertation-code/configs/cleaned_labels.gpkg


In [43]:
# test the created cleaned labels geopackage


labels_gdf = gpd.read_file(cleaned_label_fp)

labels_gdf.head()


,label_id,longitude,latitude,location,obs_date,comparison_dates_used,s2_target_image_id,s2_old_image_id,s1_image_id,class_label,notes,created_at,updated_at,created_by,class_int,lc,geometry
0,HA0001_20210121,27.823149,-25.752589,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:04.027316+00:00,2026-05-14T14:19:04.027316+00:00,bensutton,3,1,POINT (27.82315 -25.75259)
1,HA0002_20210121,27.807339,-25.760474,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:06.695977+00:00,2026-05-14T14:19:06.695977+00:00,bensutton,3,1,POINT (27.80734 -25.76047)
2,HA0003_20210121,27.809334,-25.755411,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:10.017332+00:00,2026-05-14T14:19:10.017332+00:00,bensutton,3,1,POINT (27.80933 -25.75541)
3,HA0004_20210121,27.805043,-25.758155,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:13.638342+00:00,2026-05-14T14:19:13.638342+00:00,bensutton,3,1,POINT (27.80504 -25.75816)
4,HA0005_20210121,27.819760,-25.762774,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:18.937682+00:00,2026-05-14T14:19:18.937682+00:00,bensutton,3,1,POINT (27.81976 -25.76277)


In [40]:
# Make a bounding box for each site based on the total bounds of the labelled points and add a padding

make_padded_bbox_all_location(sites_file= site_fp, project_root= PROJECT_ROOT)

Created padded bbox from the points files for Vembanad, Winam, Inle, Hartbeespoort, Mula, RawaPening, Rodman, Valsequillo


In [141]:
# Memomry limits made it neccesary to create an AWEI_p95 (with the 95th percentile for AWEIsh, 2019-2025) image that is saved to assets for each location.
# After running the code to export use the returned task list areto ensure exports are completed before running create_images_for_all_locations()

task_list = export_awei_p95_all_locations(ee_project= EE_PROJECT, sites_file= site_fp)

Deleted existing asset: projects/water-hyacinth-493214/assets/awei_p95_Vembanad
Exporting AWEIp95 asset for Vembanad → projects/water-hyacinth-493214/assets/awei_p95_Vembanad
Deleted existing asset: projects/water-hyacinth-493214/assets/awei_p95_Winam
Exporting AWEIp95 asset for Winam → projects/water-hyacinth-493214/assets/awei_p95_Winam
Deleted existing asset: projects/water-hyacinth-493214/assets/awei_p95_Inle
Exporting AWEIp95 asset for Inle → projects/water-hyacinth-493214/assets/awei_p95_Inle
Deleted existing asset: projects/water-hyacinth-493214/assets/awei_p95_Hartbeespoort
Exporting AWEIp95 asset for Hartbeespoort → projects/water-hyacinth-493214/assets/awei_p95_Hartbeespoort
Deleted existing asset: projects/water-hyacinth-493214/assets/awei_p95_Mula
Exporting AWEIp95 asset for Mula → projects/water-hyacinth-493214/assets/awei_p95_Mula
Deleted existing asset: projects/water-hyacinth-493214/assets/awei_p95_RawaPening
Exporting AWEIp95 asset for RawaPening → projects/water-hyaci

In [104]:
for task in task_list:
    print(task.status())

{'state': 'RUNNING', 'description': 'awei_p95_Vembanad', 'priority': 100, 'creation_timestamp_ms': 1780569199477, 'update_timestamp_ms': 1780569373555, 'start_timestamp_ms': 1780569202322, 'task_type': 'EXPORT_IMAGE', 'attempt': 1, 'id': 'Z2UM3XCRSJLTNFLLFTRHTLD6', 'name': 'projects/water-hyacinth-493214/operations/Z2UM3XCRSJLTNFLLFTRHTLD6'}
{'state': 'READY', 'description': 'awei_p95_Winam', 'priority': 100, 'creation_timestamp_ms': 1780569200142, 'update_timestamp_ms': 1780569202382, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'VMB6GKVYTAGYVX3Z3IP4H7TR', 'name': 'projects/water-hyacinth-493214/operations/VMB6GKVYTAGYVX3Z3IP4H7TR'}
{'state': 'READY', 'description': 'awei_p95_Inle', 'priority': 100, 'creation_timestamp_ms': 1780569200851, 'update_timestamp_ms': 1780569205192, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'ZFGXZIGVH5DUBNLA3FIPKV3S', 'name': 'projects/water-hyacinth-493214/operations/ZFGXZIGVH5DUBNLA3FIPKV3S'}
{'state': 'READY', 'description

In [155]:
image_collection = create_images_for_all_locations(sites_file= site_fp, cleaned_label_fp=cleaned_label_fp, ee_project=EE_PROJECT, clip = False)

Created image collection for: Vembanad
Created image collection for: Winam
Created image collection for: Inle
Created image collection for: Hartbeespoort
Created image collection for: Mula
Created image collection for: RawaPening
Created image collection for: Rodman
Created image collection for: Valsequillo


In [147]:
print(image_collection.first().bandNames().getInfo())

['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'AWEIp95']


In [148]:
# Due to memory limits it is not possible to convert all the sampled points to a data frame, therefore the sampled points for each location 
# is export to drive as a CSV separately


# Trying to save all without aweip95

get_samples(merged_ic=image_collection,cleaned_label_fp= cleaned_label_fp)

Exporting sampled points for Hartbeespoort to drive/Dissertation
Exporting sampled points for Mula to drive/Dissertation
Exporting sampled points for RawaPening to drive/Dissertation
Exporting sampled points for Vembanad to drive/Dissertation
Exporting sampled points for Valsequillo to drive/Dissertation
Exporting sampled points for Rodman to drive/Dissertation
Exporting sampled points for Winam to drive/Dissertation
Exporting sampled points for Inle to drive/Dissertation


In [149]:
for task in ee.batch.Task.list():
    status = task.status()
    desc = status.get("description", "")

    if desc.startswith("sampled_points_"):
        print(desc, status["state"])
        print("EECU seconds:", status.get("batch_eecu_usage_seconds"))
        if status["state"] == "FAILED":
            print(status.get("error_message"))

sampled_points_Inle READY
EECU seconds: None
sampled_points_Winam READY
EECU seconds: None
sampled_points_Rodman READY
EECU seconds: None
sampled_points_Valsequillo RUNNING
EECU seconds: 1.241310477
sampled_points_Vembanad RUNNING
EECU seconds: 0.906820774
sampled_points_RawaPening COMPLETED
EECU seconds: 32.84038543701172
sampled_points_Mula COMPLETED
EECU seconds: 78.7459487915039
sampled_points_Hartbeespoort COMPLETED
EECU seconds: 38.065589904785156
sampled_points_Inle FAILED
EECU seconds: 896.710693359
User memory limit exceeded.
sampled_points_Winam FAILED
EECU seconds: 657.202514648
User memory limit exceeded.
sampled_points_Rodman COMPLETED
EECU seconds: 13453.021484375
sampled_points_Valsequillo COMPLETED
EECU seconds: 16769.966796875
sampled_points_Vembanad FAILED
EECU seconds: 281.488739013
User memory limit exceeded.
sampled_points_RawaPening COMPLETED
EECU seconds: 14951.77734375
sampled_points_Mula FAILED
EECU seconds: 350.226226806
User memory limit exceeded.
sampled_poi

In [138]:
# Load location samples from csvs in outputs/sampled_points_by_location, combine into a single dataframe and save 

samples_outpath = PROJECT_ROOT / "outputs" / "sample_points.csv"

with open(site_fp) as f:
    sites= json.load(f)

location_list = list(sites.get("sites", {}).keys())

list_of_location_dfs = []

for location in location_list:
    location_samples_fp = PROJECT_ROOT / "outputs/sampled_points_by_location/sampled_points_no_AWEI" / f"sampled_points_{location}.csv" 

    location_df = pd.read_csv(location_samples_fp)

    list_of_location_dfs.append(location_df)

all_samples = pd.concat(list_of_location_dfs, ignore_index= True)

all_samples.to_csv(samples_outpath, index= False)



In [118]:
len(all_samples.loc[all_samples["class_int"] == 3])

7989

In [119]:
all_samples.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16019 entries, 0 to 2039
Data columns (total 20 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   system:index  16019 non-null  object 
 1   B11           16019 non-null  int64  
 2   B12           16019 non-null  int64  
 3   B2            16019 non-null  int64  
 4   B3            16019 non-null  int64  
 5   B4            16019 non-null  int64  
 6   B5            16019 non-null  int64  
 7   B6            16019 non-null  int64  
 8   B7            16019 non-null  int64  
 9   B8            16019 non-null  int64  
 10  B8A           16019 non-null  int64  
 11  class_int     16019 non-null  int64  
 12  class_label   16019 non-null  object 
 13  label_id      16019 non-null  object 
 14  latitude      16019 non-null  float64
 15  lc            16019 non-null  int64  
 16  location      16019 non-null  object 
 17  longitude     16019 non-null  float64
 18  obs_date      16019 non-null  ob

In [ ]:
mula_wh_samples = samples.loc[(samples["location"]=="mula") & (samples["lc"]== 1.0), ["B11", "B12", "B3", "B4", "B5", "B8"]]
mula_non_wh_samples = samples.loc[(samples["location"]=="mula")& (samples["lc"]==0.0), ["B11", "B12", "B3", "B4", "B5", "B8"]]
bins = np.linspace(0,8000, 20)

for col in ["B11", "B12", "B3", "B4", "B5", "B8"]:
    plt.hist(mula_wh_samples[col], bins, alpha=0.5, label='x')
    plt.hist(mula_non_wh_samples[col], bins, alpha=0.5, label='y')
    plt.legend(loc='upper right')
    plt.title(f"{col}")
    plt.show()



In [157]:
export_patches(merged_ic = image_collection, cleaned_label_fp= cleaned_label_fp, kernel_size= 15)

Exporting sampled patches for Hartbeespoort as GeoJSON to drive/Dissertation
Exporting sampled patches for Mula as GeoJSON to drive/Dissertation
Exporting sampled patches for RawaPening as GeoJSON to drive/Dissertation
Exporting sampled patches for Vembanad as GeoJSON to drive/Dissertation
Exporting sampled patches for Valsequillo as GeoJSON to drive/Dissertation
Exporting sampled patches for Rodman as GeoJSON to drive/Dissertation
Exporting sampled patches for Winam as GeoJSON to drive/Dissertation
Exporting sampled patches for Inle as GeoJSON to drive/Dissertation


In [156]:
export_patches(merged_ic = image_collection, cleaned_label_fp= cleaned_label_fp, kernel_size= 31)

Exporting sampled patches for Hartbeespoort as GeoJSON to drive/Dissertation
Exporting sampled patches for Mula as GeoJSON to drive/Dissertation
Exporting sampled patches for RawaPening as GeoJSON to drive/Dissertation
Exporting sampled patches for Vembanad as GeoJSON to drive/Dissertation
Exporting sampled patches for Valsequillo as GeoJSON to drive/Dissertation
Exporting sampled patches for Rodman as GeoJSON to drive/Dissertation
Exporting sampled patches for Winam as GeoJSON to drive/Dissertation
Exporting sampled patches for Inle as GeoJSON to drive/Dissertation


In [134]:
asset_id = 'projects/' + EE_PROJECT + '/assets/awei_p95_Winam'

AWEI_image = ee.Image(asset_id)

display('bands', AWEI_image.bandNames())

'bands'

In [ ]:
stats = AWEI_image.reduceRegion(
    reducer=ee.Reducer.minMax()
        .combine(
            reducer2=ee.Reducer.mean(),
            sharedInputs=True
        ),
    geometry=AWEI_image.geometry(),
    scale=10,      # use the appropriate resolution
    maxPixels=1e13
)

print(stats.getInfo())

{'AWEIp95_max': 12159.125, 'AWEIp95_mean': 955.1331094299671, 'AWEIp95_min': -5204}
